In [1]:
import torch
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO
from PIL import Image

import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

import os

In [3]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, split):
        self.img_dir = os.path.join(root_dir, split, 'images')
        self.label_dir = os.path.join(root_dir, split, 'labels')
        self.imgs = sorted(os.listdir(self.img_dir))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, index):
        img_path = os.path.join(self.img_dir, self.imgs[index])
        label_path = os.path.join(self.label_dir, os.path.splitext(self.imgs[index])[0] + '.txt')

        img = Image.open(img_path).convert('RGB')
        w, h = img.size

        boxes = []
        labels = []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, x_center, y_center, box_w, box_h = map(float, line.strip().split())
                    x_min = (x_center - box_w/2) * w
                    x_max = (x_center + box_w/2) * w
                    y_min = (y_center - box_h/2) * h
                    y_max = (y_center + box_h/2) * h
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {'boxes': boxes, 'labels': labels}

        return img, target


In [4]:
train_dataset = CustomDataset('data','train')

In [5]:
img, label = train_dataset[0]

In [6]:
label

{'boxes': tensor([[581.5000, 515.5000, 626.5000, 570.5000]]),
 'labels': tensor([1])}